# Robust Angular Diameters in Python (`RADPy`) : SED Fitting Tutorial

## Introduction to SED fitting

SED stands for Spectral Energy Distribution. For this purpose, SED fitting describes the process of taking photometry of a star and fitting a model stellar spectrum to that photometry. We use SED fits to determine stellar parameters such as temperature, surface gravity, metallicity, radii, and extinction. 

For interferometric analysis, you need a bolometric flux or $F_{Bol}$ in order to determine the limb darkened angular diameter. To get a bolometric flux, we use SED fits and then integrate under the curve, or model spectrum, over all wavelengths. `RADPy` is outfitted with its own SED fitting module. 

`RADPy` used the Python package `SEDFit` developed by Marina Kounkel (found here: <https://github.com/mkounkel/SEDFit>). This package requires a few additional packages:

- dust_extinction
- dustmaps
- tqdm
- tensorflow

These python packages are unfortunately only designed to work with linux and MacOS systems. If you are running Windows, please follow the instruction below on how to set up a virtual environment:

## Setting up a virtual environment for Windows

I highly recommend getting Windows Subsystem for Linux (WSL). I use WSL with Ubuntu 24.04. If you do not have WSL, follow their download and set up instructions found here: <https://ubuntu.com/desktop/wsl>. 

Once you have a WSL account, make sure you have python installed (at least version 3.12). I use conda to activate everything, so you will need to ensure conda is also installed. 

To create your virtual environment, type `conda create -n [environment name]`. It will default to the version of python that you currently have installed. To indicate which version, just add `python=[version num]` after `[environment name]`. 

Once you have the environment created, follow the install instructions for `RADPy`. This should ensure that all the packages needed will be downloaded. 

## What is in this notebook?

This tutorial is designed to show the general steps on how a user would implement `RADPy` for your own SED fitting needs for single stars. This notebook will go over how to import all the needed modules, how to create a photometry file for `RADPy`, how to read in the photometry, how to perform the SED fit, how to calculate the bolometric flux, and how to generate publication ready plots. 

## Usage

If you do use this package, please cite using the citation file located at <https://github.com/spaceashley/radpy>

In addition, make sure you also cite:

@software{sedfit,
	author = {{Kounkel}, Marina},
	doi = {10.5281/zenodo.8076500},
	month = jun,
	publisher = {Zenodo},
	title = {SEDFit},
	url = {https://doi.org/10.5281/zenodo.8076500},
	year = 2023}

## Contact

- Ashley Elliott, (aelli76@lsu.edu)

## Resources
Link to the github repository: <https://github.com/spaceashley/radpy>

Link to the papers for each model:
- BT-Settl: https://ui.adsabs.harvard.edu/abs/2011ASPC..448...91A/abstract 

- Kurucz: https://adsabs.harvard.edu/full/1992A%26A...264..557B 

- Coelho: https://ui.adsabs.harvard.edu/abs/2014MNRAS.440.1027C/abstract 

- PHOENIX: https://ui.adsabs.harvard.edu/abs/2013A%26A...553A...6H/abstract


## Step 1: Getting the data set up for use

Now that you have everything installed and ready to go, we need to first ensure the photometry is in the correct format. Your photometry file should follow the template provided on the github page for `RADPy`, entitled 'template_photometry_datafile,csv'. Please download this template and remove the filters you don't need. 

Currently, `RADPy` accepts photometry from the following:

- 2MASS J, H, Ks
- Cousins U, B, V, R, I
- GAIA G, GBp, GRp
- GALEX FUV, NUV
- Johnson U, B, V, R, I, J, K, H
- Hipparcos Hp
- PAN-STARRS PS1 g,r,i,z,y
- SLOAN SDSS u, g, r, i, z
- Spitzer IRAC 3.6, 4.5, 5.8, 8.0
- Spitzer MIPS 24micron
- Stromgren u, v, b, y
- TESS T
- TYCHO Vt, Bt
- WISE W1, W2, W3, W4
- XMM OT V, B, U, UVW1, UVW2, UVM2

If you would like another bandpass, please contact me and I can see about adding it. All of the filter bandpass information such as reference wavelength, effective width, and zero point is taken from the SVO filter service (found here: <https://svo2.cab.inta-csic.es/theory/fps/>). 

Once you have your photometry file, save it as a .csv. 

Now you have a data file ready to use!

## Step 2: Using `radpy.sedfit`

To import the module, use the following:

In [1]:
from radpy.sedfit import *

/home/oxfor/miniforge/envs/sed-env/lib/python3.12/site-packages/radpy/sedfit.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Ignore the warnings that might pop involving `pkg_resources` and 'Warning: the passwords of all user accounts have been inactivated'. These warnings relate to the `SEDFit` package and the creator is in the process of fixing them. 

Alright now lets read in our photometry.

To do so, copy the file path from where your photometry data file is located and copy and paste it into the variable `filename` below. 

The function `read_in_photometry` does a lot of things behind the scenes. It first reads in your data file as a `pandas DataFrame`. It then reads in the filter name, and pulls the corresponding filter information from the 'svo_filter_info' file, which contains the reference wavelength ($\rm \mathring{A}$), effective width ($\rm \mathring{A}$), and the zero-point value (mag). It then converts the magnitudes into fluxes and zero point corrects them. It also determines the error on the zero-point corrected flux value ($\rm \frac{erg}{s~cm^{2}~\mathring{A}}$). It then reorganizes the data into the correct format for `SEDFit`, which requires the wavelengths to be in units of $\rm \mathring{A}$, the flux to be in $\rm log_{10}(\frac{F}{\lambda})$ and in units of $ \rm \frac{erg}{s~cm^{2}~\mathring{A}}$. 

The final output of this function is an astropy.Table that follows the format:
|Index    | sed_filter  |  la     | width    | flux                 | eflux              |
|---------|-------------|---------|----------|----------------------|--------------------|
|         |             |Angstrom | Angstrom | erg/(Angstrom s cm2) |erg/(Angstrom s cm2)|




In [2]:
filename = '/mnt/c/Users/oxfor/Research/HD158259/Data/test_photometry.csv'
phot_data = read_in_photometry(filename)

Let's check and see if the data were read in properly. Check the output astropy Table to make sure everything looks correct. 

In [3]:
phot_data

index,sed_filter,la,width,flux,eflux
,,Angstrom,Angstrom,erg / (Angstrom s cm2),erg / (Angstrom s cm2)
int64,str26,float64,float64,float64,float64
0,GAIA.GAIA3.G,6217.59,2026.485,-7.335923077353823,0.02172814905702522
1,GAIA.GAIA3.Gbp,5109.71,1078.75,-7.330206751098505,0.021732059231270973
2,GAIA.GAIA3.Grp,7769.02,1462.22,-7.343814893820858,0.021754361337811196
3,2MASS.Ks,21638.61,1253.095,-8.029595397664558,0.02356724963348267
4,2MASS.J,12393.09,760.0,-7.528755744147142,0.024602186483443145
5,2MASS.H,16494.95,1205.09,-7.749136829462185,0.024060410838241827
6,TYCHO.TYCHO.B_MvB,4194.96,370.695,-7.438266603920657,0.022513133845333516
7,TYCHO.TYCHO.V_MvB,5300.19,566.775,-7.2941748642333835,0.022065092547459583


Okay now that our photometry file has been read in and reformatted, lets move to the next step.

## Step 2: Create a Stellar Params object

Aside from the photometry data, we need a `StellarParams()` object, which will allow us to store our results to use with other features of `RADPy` later on, like fitting for a limb-darkened angular diameter. See the tutorial called SingleStarTutorial, for more details on what is in the `StellarParams()` object. 

To create a `StellarParams()` object, do the following:

In [4]:
from radpy.stellar import *

star = StellarParams()

To do an SED fit with `RADPy`, you need the following input information:

- RA: in hours, minutes, seconds
- Dec: in degrees, minutes, seconds
- logg and error in logg: in cgs units
- [Fe/H] and error in [Fe/H]: in [dex]
  
Once you've gathered this information, we can add these parameters to the `StellarParams()` object.

In [5]:
logg = 4.21
dlogg = 0.33
m = 0.0
dm = 0.12
ra = "17:25:24.0552880944"
dec = "+52:47:26.469887928"
star.ra = ra
star.dec = dec
star.logg = logg
star.logg_err = dlogg
star.feh = m
star.feh_err = dm


Let's double check and make sure it worked.

In [6]:
star

fbol = None [10⁻⁸ erg/s/cm²]
fbol_err = None [10⁻⁸ erg/s/cm²]
logg = 4.21 [dex]
logg_err = 0.33 [dex]
feh = 0.0 [dex]
feh_err = 0.12 [dex]
plx = None [mas]
plx_err = None [mas]
dist = None [pc]
dist_err = None [pc]
udthetai = None [mas]
udthetai_err = None [mas]
ldthetai = None [mas]
ldthetai_err = None [mas]
udtheta = None [mas]
udtheta_err = None [mas]
ldtheta = None [mas]
ldtheta_err = None [mas]
teff = None [K]
teff_err = None [K]
lum = None [L☉]
lum_err = None [L☉]
rad = None [R☉]
rad_err = None [R☉]
ldc_R = None [ ]
ldc_K = None [ ]
ldc_H = None [ ]
ldc_J = None [ ]
ra = 17:25:24.0552880944
dec = +52:47:26.469887928

Good. We now have populated the `StellarParams()` object with some input information.

Next step is we need to get a distance. If you already have a zero-point corrected distance, you can just add it to the `StellarParams()` object using the following:

`star.dist = D`

`star.dist_err = dD`

But if you don't have a distance yet, don't worry, `RADPy` grabs it for you. To do so, use the following:

In [7]:
D, dD = distances('HD 158259', verbose = True)
star.dist = D
star.dist_err = dD

Found Gaia DR3 ID: Gaia DR3 1416050859226670848
Corrected parallax: 36.99638 [mas]
Distance: 27.02967 +/- 0.01451 [pc]


Great. Now we have all the stellar parameters we need to start an SED fit. 

## Step 3: Perform the SED fit

To perform an SED fit, `RADPy` uses `SEDFit` as described in the introduction. This package requires a few input statements from the user. You need to know which model you would like to use, what parameters you want to fit for, and the ranges of those fit parameters. 

For models, you can chose between 4 different options:

|Model     | keyword    |
|----------|------------|
|BT-SETTL  | `'btsettl'`|
|PHOENIX   | `'phoenix'`|
|Coelho    | `'coelho'` |
|Kurucz    | `'kurucz'` |

You can fit for the following parameters:

- Temperature (Teff)
- Surface gravity (logg)
- [Fe/H]
- Radius
- Extinction ($\rm A_{V} $)

It is allowed to not fit for these values, i.e. set $\rm A_{V} = 0$. If you do chose to fit for any of these parameters, you must add a range of values in which you want to have the fit search through. For example, if you are fitting for temperature, you need a range of temperature values to give the model, such as `teffrange = [5000, 7000]`. 

The inputs into the function are as follows:

|Parameter         | What is it?                                                      | Example syntax            |
|------------------|------------------------------------------------------------------|---------------------------|
|  `x`             | `astropy.Table` containing your photometry                       | `phot_data`               |
|  `star`          | `StellarParams()` object                                         |  `star`                   |
|  `initial_guess` | Array of your initial guesses for the fit.                       | `[5000, 4.5, 0.2, 0]`     |
|                  |   Even if you are fixing a parameter, set the                    |                           | 
|                  |  initial guess in the format: [Teff, logg, [Fe/H], $\rm A_{V} $] |                           |
| `model`          | Set the keyword for which model you want to use                  |  `model = 'phoenix'`      |
|  `teffrange`     | If fitting for Teff, set your range of values.                   |  `teffrange = [5000,8000]`|
|                  |  Format: [min value, max value]                                  |                           |
|  `loggrange`     | If fitting for logg, set your range of values.                   |  `loggrange = [2.5,5.0]`  |
|                  |  Format: [min value, max value]                                  |                           |
|  `fehrange`      | If fitting for [Fe/H], set your range of values.                 |  `fehrange = [-1.0, 1.0]` |
|                  |  Format: [min value, max value]                                  |                           |
|  `avrange`       | If fitting for $\rm A_{V}$, set your range of values.            |  `avrange = [0, 1]`       |
|                  |  Format: [min value, max value]                                  |                           |
|  `fitT`          | If fitting for Teff, set equal to `True`.                        |   `fitT = True`           |
|                  | Default is `False`                                               |                           |
|  `fit_logg`      | If fitting for logg, set equal to `True`.                        |   `fit_logg = True`       |
|                  | Default is `False`                                               |                           |
|  `fit_feh`       | If fitting for [Fe/H], set equal to `True`.                      |   `fit_feh = True`        |
|                  | Default is `False`                                               |                           |
|  `fit_av`        | If fitting for $\rm A_{V}$, set equal to `True`.                 |   `fit_av = True`         |
|                  | Default is `False`                                               |                           |
| `verbose`        | If True, returns print statements.                               |    `verbose = True`       |
|                  | Useful for debugging. Default is set to `False`                  |                           |

Lets run a fit. 

(Warnings might appear about cude drivers. Ignore these. The fit works without them.)

In [8]:
sed_fit = fit_sed(phot_data, star,  [5850, 4.21, 0.0, 0.0],  model = 'phoenix', teffrange = [5000,7000], fitT = True, verbose = True)

2026-01-21 16:44:42.979797: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-21 16:44:43.037796: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-21 16:44:44.753185: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-21 16:44:45.058714: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Chi squared: 23.15039008212447
Chi squared reduced: 1.2861327823402484
Distance: 27.02967 pc
AV: 0.0 mag
Radius: [np.float64(1.2603182648920521)] Rsun
Teff: [np.float64(5800.731636163079)] K
Log g: [4.21] 
Fe/H: 0.0


The results of the fit shown above include the $\chi^{2}$, $\chi^{2}_{red}$ and the fitted values from the SED. `RADPy` also stores the SED fitted parameters automatically into the `StellarParams()` object just for comparison purposes later on. 

## Step 4: Compute the Bolometric Flux

To calculate the bolometric flux, `RADPy` uses one function called `calc_fbol`. The integration range is between the minimum model wavelength value to 100 $\mu m$. 

To call the function, you must give the function the `astropy.Table` object created by the `fit_sed` function, the `StellarParams()` object, and a unit keyword (either `'micron'` or `'AA'`). If you would like the function to pront out the final result, flag the `verbose` as `True`.

Now let's calculate.

In [9]:
fbol, fbol_err = calc_fbol(star, sed_fit, 'AA', verbose = True)


NameError: name 'model_w' is not defined

And now we have a bolometric flux!!

## Step 5: Plotting the SED fit

`RADPy` will also plot your SED on a publication ready graphic. The plotting function is called `plot_sed`. It has a few inputs to be aware of.

|Input            |   What is it?                                        | Example syntax                    |
|-----------------|------------------------------------------------------|-----------------------------------|
|  `sed_fit`      |  Your fit object created after running `fit_sed`     | `sed_fit`                         |
|  `unit`         | What wavelength unit you want your plot in           |   `'AA'`                          |
|                 | Options are Angstroms (`'AA'`) or microns `'micron'` |                                   |
| `logplot`       | Set True if you want your plot in log space          |   `logplot = True`                |
|                 | Default is True                                      |                                   |
| `fbol_lam`      | Set True if you want your flux to be multiplied      | `fbol_lam = True`                 |
|                 | by the wavelength, Default is True.                  |                                   |
| `set_axis`      | Set the axis range of your plot                      | `set_axis = [3.5, 4.5, -8.5, -7]` |
|                 | Needs to be in the correct space for the plot        |                                   |
|                 | If nothing is set, `RADPy` will set it for you       |                                   |
|                 | based on the model ranges                            |                                   |
| `title`         | Allows user to add a title to the plot.              | `title = 'HD # SED fit'`          |
| `savefig`       | Allows user to save the figure.                      | `savefig = 'HD#SEDfit.png`        |
| `show`          | Displays the figure to you. Default is True.         | `show = True`                     |

Let's plot.

In [ ]:
set_axis = [3.5,4.5, -8.5,-7]
plot_sed(sed_fit, 'AA', logplot = True, fbol_lam = True, set_axis = set_axis, title = None, savefig = None, show = True)